In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from PIL import Image

def process_ogle_lightcurves():
    # 1. Configuración de Rutas
    input_dir = r"C:\Users\santi\Downloads\Be_GSEP\Be_GSEP"
    output_dir = r"C:\Users\santi\Downloads\Be_GSEP\plots"
    results_csv = r"C:\Users\santi\Downloads\Be_GSEP\results_besc.csv"
    
    # Lista de IDs que queremos RESALTAR visualmente
    estrellas_destacadas = [
        "LMC562.02.7937", "LMC562.12.10123", "LMC562.13.11357", 
        "LMC562.25.11162", "LMC562.32.173", "LMC563.04.477", 
        "LMC563.30.7056", "LMC570.14.103", "LMC570.26.70"
    ]
    
    freq_min = 0.0011
    freq_max = 10.0
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Buscar todos los archivos .dat y .txt
    search_pattern_dat = os.path.join(input_dir, "*.dat")
    search_pattern_txt = os.path.join(input_dir, "*.txt")
    all_files = glob.glob(search_pattern_dat) + glob.glob(search_pattern_txt)
    
    if not all_files:
        print(f"No se encontraron archivos en la carpeta:\n{input_dir}")
        return

    print(f"¡Se encontraron {len(all_files)} archivos para procesar!\n")
    summary_results = []

    # 2. Bucle principal
    for file_path in all_files:
        filename = os.path.basename(file_path)
        star_id, _ = os.path.splitext(filename)
        
        print(f"Procesando {star_id}...")
                
        try:
            data = pd.read_csv(file_path, sep=r'\s+', names=['HJD', 'Mag', 'Err'], comment='#')
        except Exception as e:
            print(f"  -> Error al leer {filename}: {e}")
            continue

        if len(data) < 10:
            print(f"  -> Muy pocos datos. Saltando.")
            continue

        time = data['HJD'].values
        mag = data['Mag'].values
        err = data['Err'].values

        # 3. Limpieza de Outliers (MAD)
        median_mag = np.median(mag)
        mad_mag = np.median(np.abs(mag - median_mag)) 
        
        if mad_mag == 0: continue
        
        valid_mask = np.abs(mag - median_mag) <= (3 * mad_mag)
        time_clean = time[valid_mask]
        mag_clean = mag[valid_mask]
        err_clean = err[valid_mask]

        # 4. Lomb-Scargle
        ls = LombScargle(time_clean, mag_clean, dy=err_clean)
        frequency, power = ls.autopower(minimum_frequency=freq_min, maximum_frequency=freq_max)
        
        best_freq = frequency[np.argmax(power)]
        best_period = 1.0 / best_freq
        max_power = np.max(power)
        fap_1_percent = ls.false_alarm_level([0.01])[0]
        
        # =========================================================================
        # NUEVO: CÁLCULO DEL COEFICIENTE DE DETERMINACIÓN (R^2)
        # =========================================================================
        # Evaluamos el modelo sinusoidal de Lomb-Scargle en los tiempos observados
        mag_pred = ls.model(time_clean, best_freq)
        
        ss_res = np.sum((mag_clean - mag_pred) ** 2)
        ss_tot = np.sum((mag_clean - np.mean(mag_clean)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot != 0 else 0.0
        # =========================================================================
        
        # Se añade 'R2' al diccionario de resultados para exportar al CSV
        summary_results.append({
            'ID': star_id, 
            'Best_Period_Days': best_period,
            'Max_Power': max_power, 
            'FAP_Value': fap_1_percent,
            'R2': r2
        })

        # ==========================================
        # 5. CONFIGURACIÓN VISUAL (EL RESALTADO)
        # ==========================================
        is_highlighted = star_id in estrellas_destacadas
        
        # Definimos colores según si es destacada o no
        marker_color = '#e6550d' if is_highlighted else 'black' # Naranja si es destacada
        bg_color = '#fff5eb' if is_highlighted else 'white'     # Fondo crema si es destacada
        
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 15))
        fig.patch.set_facecolor(bg_color) # Aplicar el color de fondo a la imagen
        
        # Título especial
        if is_highlighted:
            fig.suptitle(f' [DESTACADA] Análisis de Candidato: {star_id} ', fontsize=18, fontweight='bold', color='#e6550d')
        else:
            fig.suptitle(f'Análisis de Candidato: {star_id}', fontsize=16)

        # Gráfico 1: Curva de Luz Original
        ax1.errorbar(time_clean, mag_clean, yerr=err_clean, fmt='.', color=marker_color, ecolor='gray', alpha=0.5, capsize=0)
        ax1.set_title('Curva de Luz Limpia')
        ax1.set_xlabel('HJD (días)')
        ax1.set_ylabel('Magnitud I')
        ax1.invert_yaxis()
        ax1.set_facecolor(bg_color)

        # Gráfico 2: Periodograma
        ax2.plot(frequency, power, color=marker_color, lw=1)
        ax2.axhline(fap_1_percent, color='red', linestyle='--', label='1% FAP Level')
        ax2.plot(best_freq, max_power, 'ro' if not is_highlighted else 'bo', label=f'Mejor Frec: {best_freq:.4f} d⁻¹\n(Periodo: {best_period:.4f} d)')
        ax2.set_title('Periodograma Lomb-Scargle')
        ax2.set_xlabel('Frecuencia (d⁻¹)')
        ax2.set_ylabel('Poder')
        ax2.legend()
        ax2.set_facecolor(bg_color)

        # Gráfico 3: Curva de Fase
        phase = (time_clean % best_period) / best_period
        ax3.plot(np.concatenate([phase, phase + 1]), 
                 np.concatenate([mag_clean, mag_clean]), '.', color=marker_color, alpha=0.5)
        ax3.set_title(f'Curva Plegada por Fase (P = {best_period:.4f} días)')
        ax3.set_xlabel('Fase')
        ax3.set_ylabel('Magnitud I')
        ax3.invert_yaxis()
        ax3.set_facecolor(bg_color)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        # Para que el fondo pintado se guarde correctamente
        plot_path = os.path.join(output_dir, f"{star_id}_analysis.png")
        plt.savefig(plot_path, facecolor=fig.get_facecolor(), edgecolor='none')
        plt.close()

    # 6. Exportar CSV
    if summary_results:
        results_df = pd.DataFrame(summary_results)
        results_df.to_csv(results_csv, index=False)
        print(f"\nResumen CSV guardado en '{results_csv}'.")

# ==========================================
# 7. FUNCIÓN PARA COMPILAR EL PDF FINAL
# ==========================================
def export_plots_to_pdf():
    output_dir = r"C:\Users\santi\Downloads\Be_GSEP\plots"
    pdf_output_path = r"C:\Users\santi\Downloads\Be_GSEP\reporte_completo_besc.pdf"
    
    img_files = [f for f in os.listdir(output_dir) if f.endswith('.png')]
    img_files.sort()
    
    if not img_files: return

    print("\nGenerando PDF con todos los gráficos...")
    images = []
    for filename in img_files:
        img_path = os.path.join(output_dir, filename)
        img = Image.open(img_path).convert('RGB')
        images.append(img)
    
    images[0].save(pdf_output_path, save_all=True, append_images=images[1:])
    print(f"¡ÉXITO! PDF final guardado en: {pdf_output_path}")

if __name__ == "__main__":
    process_ogle_lightcurves()
    export_plots_to_pdf()

¡Se encontraron 50 archivos para procesar!

Procesando LMC562.01.211...
Procesando LMC562.01.7994...
Procesando LMC562.02.7937...
Procesando LMC562.02.8135...
Procesando LMC562.03.8441...
Procesando LMC562.04.125...
Procesando LMC562.06.10895...
Procesando LMC562.07.11068...
Procesando LMC562.09.110...
Procesando LMC562.11.87...
Procesando LMC562.11.9588...
Procesando LMC562.12.10123...
Procesando LMC562.13.103...
Procesando LMC562.13.106...
Procesando LMC562.13.11357...
Procesando LMC562.13.11442...
Procesando LMC562.13.11454...
Procesando LMC562.14.10726...
Procesando LMC562.14.124...
Procesando LMC562.15.11956...
Procesando LMC562.15.132...
Procesando LMC562.16.12173...
Procesando LMC562.16.231...
Procesando LMC562.19.8354...
Procesando LMC562.20.78...
Procesando LMC562.20.85...
Procesando LMC562.20.9119...
Procesando LMC562.21.180...
Procesando LMC562.24.11360...
Procesando LMC562.24.11487...
Procesando LMC562.24.132...
Procesando LMC562.25.11162...
Procesando LMC562.26.8110...
Pro

In [3]:
import os
import glob
from typing import Tuple, Dict, Any

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.timeseries import LombScargle
from tqdm import tqdm

from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ExpSineSquared
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# CONFIGURACIÓN GLOBAL (HIPERPARÁMETROS)
# ==========================================
CONFIG = {
    'freq_min': 1 / 1500.0,
    'freq_max': 1 / 50.0,
    'outlier_method': 'mad',
    'mad_sigma': 3.0,
    'gp_bins': 150
}

# ==========================================
# CLASE PRINCIPAL DEL ANALIZADOR
# ==========================================
class MiraAnalyzer:
    """Encapsula la lógica de análisis fotométrico para una estrella tipo Mira."""
    
    def __init__(self, t_raw: np.ndarray, y_raw: np.ndarray, dy_raw: np.ndarray, star_name: str):
        self.star_name = star_name
        self.t_raw = t_raw
        self.y_raw = y_raw
        self.dy_raw = dy_raw
        
        self.t = np.array([])
        self.y = np.array([])
        self.dy = np.array([])
        self.mask_valid = np.array([])
        self.results: Dict[str, Any] = {}
        
    def limpiar_fotometria(self):
        if CONFIG['outlier_method'] == 'mad':
            mediana = np.median(self.y_raw)
            mad = np.median(np.abs(self.y_raw - mediana))
            limite = CONFIG['mad_sigma'] * 1.4826 * mad
            self.mask_valid = np.abs(self.y_raw - mediana) < limite
            
        elif CONFIG['outlier_method'] == 'hdbscan':
            X = np.column_stack((self.t_raw, self.y_raw))
            X_scaled = StandardScaler().fit_transform(X)
            clusterer = HDBSCAN(min_cluster_size=10, cluster_selection_epsilon=0.5)
            self.mask_valid = (clusterer.fit_predict(X_scaled) != -1)
            
            if np.sum(self.mask_valid) < len(self.t_raw) * 0.5:
                self.mask_valid = np.ones(len(self.t_raw), dtype=bool)

        self.t = self.t_raw[self.mask_valid]
        self.y = self.y_raw[self.mask_valid]
        self.dy = self.dy_raw[self.mask_valid]

    def _calcular_pdm(self, frecuencias: np.ndarray, n_bins: int = 10) -> float:
        varianzas_fase = np.zeros(len(frecuencias))
        var_total = np.var(self.y)
        for k, freq in enumerate(frecuencias):
            periodo = 1.0 / freq
            fase = (self.t % periodo) / periodo
            bin_idx = np.clip((fase * n_bins).astype(int), 0, n_bins - 1)
            
            conteo_bins = np.bincount(bin_idx, minlength=n_bins)
            suma_y_bins = np.bincount(bin_idx, weights=self.y, minlength=n_bins)
            
            validos = conteo_bins > 1
            medias = np.zeros(n_bins)
            medias[validos] = suma_y_bins[validos] / conteo_bins[validos]
            
            medias_mapeadas = medias[bin_idx]
            var_bins = np.sum((self.y - medias_mapeadas)**2 * validos[bin_idx])
            varianzas_fase[k] = (var_bins / len(self.y)) / var_total
            
        return 1.0 / frecuencias[np.argmin(varianzas_fase)]

    def _calcular_ce(self, frecuencias: np.ndarray, n_bins: int = 10) -> float:
        entropias = np.zeros(len(frecuencias))
        for k, freq in enumerate(frecuencias):
            periodo = 1.0 / freq
            fase = (self.t % periodo) / periodo
            H_matrix, _, _ = np.histogram2d(fase, self.y, bins=[n_bins, n_bins])
            p_ij = H_matrix / np.sum(H_matrix)
            p_i = np.sum(p_ij, axis=1)
            
            p_i_expandido = np.repeat(p_i, n_bins).reshape(n_bins, n_bins)
            mask = (p_ij > 0) & (p_i_expandido > 0)
            entropias[k] = -np.sum(p_ij[mask] * np.log(p_ij[mask] / p_i_expandido[mask]))
            
        return 1.0 / frecuencias[np.argmin(entropias)]

    def buscar_periodo(self):
        ls = LombScargle(self.t, self.y, self.dy)
        
        # LS Rápido (O(N log N))
        frecuencias_ls, potencia_ls = ls.autopower(minimum_frequency=CONFIG['freq_min'], 
                                                   maximum_frequency=CONFIG['freq_max'], 
                                                   samples_per_peak=10)
        idx_max = np.argmax(potencia_ls)
        freq_max = frecuencias_ls[idx_max]
        p_ls = 1.0 / freq_max
        
        # Estimación de error (FWHM aprox)
        media_potencia = np.max(potencia_ls) / 2.0
        picos_anchura = np.where(potencia_ls > media_potencia)[0]
        if len(picos_anchura) > 1:
            delta_f = np.abs(frecuencias_ls[picos_anchura[-1]] - frecuencias_ls[picos_anchura[0]])
            error_p_ls = (1.0 / freq_max**2) * delta_f
        else:
            error_p_ls = 0.0

        frecuencias_densas = np.linspace(CONFIG['freq_min'], CONFIG['freq_max'], 3000)
        p_pdm = self._calcular_pdm(frecuencias_densas)
        p_ce = self._calcular_ce(frecuencias_densas)

        # Arbitraje (String Length)
        candidatos = [p_ls, p_pdm, p_ce]
        scores = []
        for p in candidatos:
            fase = (self.t % p) / p
            y_ord = self.y[np.argsort(fase)]
            scores.append(np.sum(np.diff(y_ord)**2) / np.var(self.y))
            
        self.results['Period_final'] = candidatos[np.argmin(scores)]
        self.results['Period_Error'] = error_p_ls
        self.results['P_LS'] = p_ls
        self.results['P_PDM'] = p_pdm
        self.results['P_CE'] = p_ce
        self.results['Fase'] = (self.t % self.results['Period_final']) / self.results['Period_final']

    def ajustar_gp(self):
        fase_doble = np.concatenate([self.results['Fase'], self.results['Fase'] + 1.0])
        y_doble = np.concatenate([self.y, self.y])
        dy_doble = np.concatenate([self.dy, self.dy])
        
        n_bins = CONFIG['gp_bins']
        bin_edges = np.linspace(0, 2, n_bins + 1)
        bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
        bin_idx = np.digitize(fase_doble, bin_edges) - 1
        
        binned_phase, binned_y, binned_dy = [], [], []
        for i in range(n_bins):
            mask = (bin_idx == i)
            if np.sum(mask) > 0:
                pesos = 1.0 / (dy_doble[mask]**2)
                binned_phase.append(bin_centers[i])
                binned_y.append(np.average(y_doble[mask], weights=pesos))
                binned_dy.append(np.sqrt(1.0 / np.sum(pesos)))
                
        f_train = np.array(binned_phase).reshape(-1, 1)
        y_train = np.array(binned_y)
        dy_train = np.array(binned_dy)

        # Kernel Híbrido Estabilizado
        kernel_periodico = ExpSineSquared(length_scale=1.0, periodicity=1.0, periodicity_bounds="fixed")
        kernel_rbf = RBF(length_scale=0.2, length_scale_bounds=(0.05, 1.0))
        kernel_ruido = WhiteKernel(noise_level=np.median(dy_train)**2, noise_level_bounds=(1e-6, 1e-1))
        
        kernel_hibrido = (1.0 * kernel_periodico * kernel_rbf) + kernel_ruido
        
        gp = GaussianProcessRegressor(kernel=kernel_hibrido, n_restarts_optimizer=2, normalize_y=True)
        gp.fit(f_train, y_train)

        y_pred_eval = gp.predict(self.results['Fase'].reshape(-1, 1))
        self.results['R2'] = r2_score(self.y, y_pred_eval)
        self.results['RMSE'] = np.sqrt(mean_squared_error(self.y, y_pred_eval))

        fase_grid = np.linspace(0, 2, 300).reshape(-1, 1)
        y_smooth, sigma_smooth = gp.predict(fase_grid, return_std=True)
        self.results['Visual'] = (f_train, y_train, fase_grid, y_smooth, sigma_smooth)

    def ejecutar_pipeline(self):
        self.limpiar_fotometria()
        self.buscar_periodo()
        self.ajustar_gp()
        return self

# ==========================================
# GESTOR SECUENCIAL PARA JUPYTER
# ==========================================
def procesar_archivo(archivo: str) -> MiraAnalyzer:
    nombre = os.path.basename(archivo).replace('.csv', '')
    df = pd.read_csv(archivo)
    analyzer = MiraAnalyzer(df['t'].values, df['y'].values, df['dy'].values, nombre)
    return analyzer.ejecutar_pipeline()

def generar_reportes_seguros(carpeta_entrada: str, archivo_pdf_salida: str, archivo_csv_salida: str):
    archivos = glob.glob(os.path.join(carpeta_entrada, '*.csv'))
    if not archivos:
        print(f"No se encontraron archivos en: {carpeta_entrada}")
        return

    plt.rcParams.update({'font.family': 'serif', 'xtick.direction': 'in', 'ytick.direction': 'in'})
    datos_exportacion = []
    
    print("Modo Jupyter detectado. Iniciando procesamiento secuencial...")

    # Procesamiento estrella por estrella (a prueba de fallos)
    analizadores_terminados = []
    for arc in tqdm(archivos, desc="Estrellas Modeladas"):
        analizadores_terminados.append(procesar_archivo(arc))

    # Escritura y gráficas
    with PdfPages(archivo_pdf_salida) as pdf:
        for analyzer in tqdm(analizadores_terminados, desc="Generando PDF y CSV"):
            
            datos_exportacion.append({
                'Star_Name': analyzer.star_name,
                'Period_LS_days': round(analyzer.results['P_LS'], 2),
                'Period_PDM_days': round(analyzer.results['P_PDM'], 2),
                'Period_CE_days': round(analyzer.results['P_CE'], 2),
                'Best_Period_days': round(analyzer.results['Period_final'], 2),
                'Period_Error_days': round(analyzer.results['Period_Error'], 3),
                'R2_Score': round(analyzer.results['R2'], 3),
                'RMSE_mag': round(analyzer.results['RMSE'], 3)
            })

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [1, 1]})
            
            ax1.plot(analyzer.t_raw[~analyzer.mask_valid], analyzer.y_raw[~analyzer.mask_valid], 'x', color='red', alpha=0.3, markersize=3, label='Outliers')
            ax1.plot(analyzer.t, analyzer.y, '.', color='navy', markersize=3, label='Datos limpios')
            ax1.invert_yaxis()
            ax1.set_xlabel('Modified Julian Day')
            ax1.set_ylabel('Magnitude')
            ax1.set_title(analyzer.star_name, loc='left', fontsize=10)
            ax1.legend(loc='best', fontsize=8)
            
            fase = analyzer.results['Fase']
            f_train, y_train, fase_grid, y_smooth, sigma_smooth = analyzer.results['Visual']
            
            ax2.plot(np.concatenate([fase, fase + 1]), np.concatenate([analyzer.y, analyzer.y]), '.', color='gray', alpha=0.3, markersize=2)
            ax2.plot(f_train, y_train, 's', color='darkorange', markersize=3, label='Binned')
            ax2.plot(fase_grid, y_smooth, 'b-', linewidth=2, label='GP Fit')
            ax2.fill_between(fase_grid.ravel(), y_smooth - 2*sigma_smooth, y_smooth + 2*sigma_smooth, color='blue', alpha=0.2)
            ax2.invert_yaxis()
            ax2.set_xlabel('Phase')
            
            p_final = analyzer.results['Period_final']
            p_err = analyzer.results['Period_Error']
            texto_stats = (f"LS: {analyzer.results['P_LS']:.1f} d\nPDM: {analyzer.results['P_PDM']:.1f} d\nCE: {analyzer.results['P_CE']:.1f} d\n"
                           f"P_final: {p_final:.1f} ± {p_err:.1f} d\n"
                           f"R²: {analyzer.results['R2']:.2f} | RMSE: {analyzer.results['RMSE']:.2f}")
            
            ax2.text(0.95, 0.05, texto_stats, transform=ax2.transAxes, ha='right', va='bottom', 
                     fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'))
            
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    # Guardar CSV final
    df_resultados = pd.DataFrame(datos_exportacion)
    df_resultados.to_csv(archivo_csv_salida, index=False)
    print(f"\n¡Listo! Análisis completo y guardado correctamente.")

# ==========================================
# EJECUCIÓN 
# ==========================================
if __name__ == '__main__':
    carpeta_datos = r"C:\Users\santi\Downloads\selected_mira_timeseries (1)\data_selected"
    nombre_pdf = r"C:\Users\santi\Downloads\Curvas_de_Luz_Miras_v3.pdf"
    nombre_csv = r"C:\Users\santi\Downloads\Resultados_Periodos_Miras_v3.csv"
    
    print("==================================================")
    print("   PIPELINE ASTROFÍSICO V3 - MODO SEGURO JUPYTER")
    print("==================================================")
    
    generar_reportes_seguros(carpeta_datos, nombre_pdf, nombre_csv)

   PIPELINE ASTROFÍSICO V3 - MODO SEGURO JUPYTER
Modo Jupyter detectado. Iniciando procesamiento secuencial...


Estrellas Modeladas:   0%|          | 0/40 [00:00<?, ?it/s]c:\Users\santi\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
Estrellas Modeladas:   2%|▎         | 1/40 [00:04<02:44,  4.21s/it]c:\Users\santi\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
Estrellas Modeladas:   5%|▌         | 2/40 [00:07<02:24,  3.81s/it]c:\Users\santi\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the s


¡Listo! Análisis completo y guardado correctamente.
